# E-Commerce Customer Behaviour Analysis

**Internship Project**

This notebook analyzes customer purchasing behaviour on an e-commerce platform using transaction-level data. It covers:

1. Data loading & cleaning
2. Exploratory Data Analysis (EDA)
3. Customer segmentation using **RFM Analysis** (Recency, Frequency, Monetary)
4. K-Means clustering on RFM scores
5. Cohort retention analysis
6. Category & payment method insights
7. Key business recommendations

**Dataset:** Synthetic e-commerce transaction data (`data/ecommerce_transactions.csv`) generated to resemble real-world customer purchase patterns, including realistic messiness (missing values, duplicates) for cleaning practice.


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/ecommerce_transactions.csv", parse_dates=["transaction_date", "signup_date"])
print(df.shape)
df.head()


In [ ]:
df.info()


## 2. Data Cleaning

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isna().sum())

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")


In [ ]:
# Drop exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Fill missing device_type with mode (most common device)
df["device_type"] = df["device_type"].fillna(df["device_type"].mode()[0])

# Rating: keep NaN as "not rated" -- create explicit flag instead of imputing a fake score
df["is_rated"] = df["rating"].notna().astype(int)

print("Post-cleaning shape:", df.shape)
print("Remaining missing values:\n", df.isna().sum())


In [ ]:
# Feature engineering
df["order_month"] = df["transaction_date"].dt.to_period("M").astype(str)
df["order_weekday"] = df["transaction_date"].dt.day_name()
df["order_hour"] = df["transaction_date"].dt.hour
df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 55, 65],
                          labels=["18-25", "26-35", "36-45", "46-55", "56-65"])

df.head()


## 3. Exploratory Data Analysis

### 3.1 Revenue Overview

In [ ]:
total_revenue = df["final_amount"].sum()
total_orders = df["transaction_id"].nunique()
total_customers = df["customer_id"].nunique()
avg_order_value = df["final_amount"].mean()

print(f"Total Revenue: ₹{total_revenue:,.0f}")
print(f"Total Orders: {total_orders:,}")
print(f"Unique Customers: {total_customers:,}")
print(f"Average Order Value: ₹{avg_order_value:,.2f}")


In [ ]:
monthly_revenue = df.groupby("order_month")["final_amount"].sum().sort_index()

plt.figure(figsize=(12, 5))
monthly_revenue.plot(kind="line", marker="o", color="#4C72B0")
plt.title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../images/monthly_revenue_trend.png", dpi=150)
plt.show()


### 3.2 Category Performance

In [ ]:
category_stats = df.groupby("category").agg(
    orders=("transaction_id", "nunique"),
    revenue=("final_amount", "sum"),
    avg_rating=("rating", "mean")
).sort_values("revenue", ascending=False)

category_stats


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

category_stats["revenue"].sort_values().plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Revenue by Category")
axes[0].set_xlabel("Revenue (₹)")

category_stats["orders"].sort_values().plot(kind="barh", ax=axes[1], color="#C44E52")
axes[1].set_title("Order Count by Category")
axes[1].set_xlabel("Number of Orders")

plt.tight_layout()
plt.savefig("../images/category_performance.png", dpi=150)
plt.show()


### 3.3 Payment Method & Device Preferences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df["payment_method"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[0], cmap="viridis")
axes[0].set_title("Payment Method Share")
axes[0].set_ylabel("")

df["device_type"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[1], cmap="magma")
axes[1].set_title("Device Type Share")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../images/payment_device_share.png", dpi=150)
plt.show()


### 3.4 Purchase Timing Patterns

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales = df.groupby("order_weekday")["final_amount"].sum().reindex(weekday_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=weekday_sales.index, y=weekday_sales.values, color="#8172B2")
plt.title("Revenue by Day of Week")
plt.ylabel("Revenue (₹)")
plt.xlabel("")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("../images/revenue_by_weekday.png", dpi=150)
plt.show()


### 3.5 Returns & Ratings

In [ ]:
return_rate = df["returned"].mean() * 100
print(f"Overall return rate: {return_rate:.2f}%")

plt.figure(figsize=(8, 5))
sns.countplot(data=df.dropna(subset=["rating"]), x="rating", hue="returned", palette="Set2")
plt.title("Return Behaviour vs Rating")
plt.xlabel("Rating")
plt.ylabel("Order Count")
plt.legend(title="Returned", labels=["No", "Yes"])
plt.tight_layout()
plt.savefig("../images/returns_vs_rating.png", dpi=150)
plt.show()


## 4. RFM Analysis (Recency, Frequency, Monetary)

RFM segmentation is one of the most widely used techniques in customer behaviour analysis:

- **Recency (R):** How recently did the customer purchase?
- **Frequency (F):** How often do they purchase?
- **Monetary (M):** How much do they spend?


In [ ]:
snapshot_date = df["transaction_date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("customer_id").agg(
    recency=("transaction_date", lambda x: (snapshot_date - x.max()).days),
    frequency=("transaction_id", "nunique"),
    monetary=("final_amount", "sum")
).reset_index()

rfm.head()


In [ ]:
# Score each dimension into quartiles (1 = worst, 4 = best)
rfm["R_score"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

def segment_customer(score):
    if score >= 10:
        return "Champions"
    elif score >= 8:
        return "Loyal Customers"
    elif score >= 6:
        return "Potential Loyalists"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost Customers"

rfm["segment"] = rfm["RFM_score"].apply(segment_customer)
rfm.head()


In [ ]:
segment_counts = rfm["segment"].value_counts()

plt.figure(figsize=(9, 6))
sns.barplot(x=segment_counts.values, y=segment_counts.index, palette="viridis")
plt.title("Customer Segments (RFM)", fontsize=14, fontweight="bold")
plt.xlabel("Number of Customers")
plt.tight_layout()
plt.savefig("../images/rfm_segments.png", dpi=150)
plt.show()


In [ ]:
segment_summary = rfm.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
).sort_values("avg_monetary", ascending=False)

segment_summary


## 5. K-Means Clustering on RFM

As a complement to rule-based RFM segmentation, we apply K-Means clustering on standardized RFM values to let the data define natural groupings.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

rfm_features = rfm[["recency", "frequency", "monetary"]]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

# Elbow method to choose k
inertias = []
K_range = range(1, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker="o", color="#4C72B0")
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.savefig("../images/elbow_method.png", dpi=150)
plt.show()


In [ ]:
# Fit final model (k=4 chosen from elbow curve)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["cluster"] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby("cluster").agg(
    customers=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
)
cluster_summary


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection="3d")

scatter = ax.scatter(rfm["recency"], rfm["frequency"], rfm["monetary"],
                      c=rfm["cluster"], cmap="viridis", s=40, alpha=0.7)

ax.set_xlabel("Recency (days)")
ax.set_ylabel("Frequency (orders)")
ax.set_zlabel("Monetary (₹)")
ax.set_title("Customer Clusters based on RFM", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("../images/rfm_clusters_3d.png", dpi=150)
plt.show()


## 6. Cohort Retention Analysis

We group customers by the month they signed up (their **cohort**) and track what fraction of each cohort keeps purchasing in subsequent months.


In [ ]:
df["signup_month"] = df["signup_date"].dt.to_period("M")
df["order_period"] = df["transaction_date"].dt.to_period("M")
df["cohort_index"] = (df["order_period"] - df["signup_month"]).apply(lambda x: x.n)

cohort_data = df.groupby(["signup_month", "cohort_index"])["customer_id"].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index="signup_month", columns="cohort_index", values="customer_id")

cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0)

retention_display = retention.iloc[:12, :12]  # first 12 cohorts, first 12 months
retention_display.round(2)


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(retention_display, annot=True, fmt=".0%", cmap="Blues", cbar_kws={"label": "Retention Rate"})
plt.title("Monthly Cohort Retention Heatmap", fontsize=14, fontweight="bold")
plt.xlabel("Months Since Signup")
plt.ylabel("Signup Cohort")
plt.tight_layout()
plt.savefig("../images/cohort_retention_heatmap.png", dpi=150)
plt.show()


## 7. Key Insights & Business Recommendations

Based on the analysis above:

1. **Revenue concentration:** A small share of customers (the "Champions" segment) drive a disproportionate share of revenue — prioritize loyalty rewards and early access for this group.
2. **At-risk & lost customers:** A meaningful portion of the base has high recency values (haven't purchased recently). Targeted win-back email/discount campaigns are recommended.
3. **Category strategy:** Electronics and Fashion drive the most revenue; bundling promotions across these categories with lower-performing categories (e.g. Books, Toys) could lift overall basket size.
4. **Mobile-first experience:** The majority of transactions come from mobile devices — continued investment in mobile UX and app-exclusive deals is likely to have outsized impact.
5. **Returns correlate with low ratings:** Orders rated 1–2 stars have a much higher return rate. Improving product descriptions/images and post-purchase support for low-rated categories could reduce returns.
6. **Retention drop-off:** Cohort analysis shows retention typically declines sharply after month 1–2. A structured onboarding/re-engagement flow in the first 60 days after signup could improve long-term retention.

---
*This analysis was built as part of a Data Analytics / Business Analytics internship project, demonstrating skills in data cleaning, EDA, customer segmentation (RFM), unsupervised learning (K-Means), and cohort retention analysis using Python (pandas, seaborn, scikit-learn).*
